# CelebA-Spoof × SCRFD — Crop Factor & Minimum Face Size Analysis v2

Notebook này là bản sửa của test SCRFD cũ, tập trung vào **hai quyết định preprocessing trước khi train PAD**:

1. **SCRFD crop factor nào tạo context gần nhất với crop train CelebA hiện tại?**
2. **Nên loại face nhỏ hơn bao nhiêu pixel khỏi TRAIN để tránh upsample/noise quá mạnh?**

## Nguyên tắc phương pháp

Reference hiện tại của E1:

```text
CelebA _BB.txt → square crop → expansion 1.5×
```

Pipeline mới dự kiến:

```text
Raw image → SCRFD bbox → square crop → SCRFD expansion factor = ?
```

Factor chỉ được ước lượng trên **reliable matched subset** có raw-IoU đủ cao và center-shift đủ thấp, để annotation outlier không kéo sai kết quả.

Official Test **không bị loại face nhỏ**. Face-size threshold chỉ dùng khi tạo TRAIN manifest.

Mặc định dùng `ANALYSIS_SPLIT = "train"` vì crop factor và min-face threshold là preprocessing hyperparameters; không nên chọn chúng bằng official Test.

---

## Compact-output mode

Bản này được tối ưu cho Kaggle:

- không render DataFrame lớn inline;
- không render gallery ảnh mặc định;
- plot được lưu vào `/kaggle/working/scrfd_analysis_plots/`;
- CSV/JSON đầy đủ vẫn được lưu;
- notebook chỉ in các summary cần thiết để chọn crop factor và min-face threshold.

Nếu thật sự cần xem gallery sau khi có kết quả, đổi:

```python
RENDER_GALLERIES = True
```

rồi chỉ chạy lại các cell gallery.


In [2]:
!pip install -q insightface onnxruntime pandas

import os, json, math, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from insightface.app import FaceAnalysis

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("OpenCV:", cv2.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 36.0 MB/s eta 0:00:0000:0100:01
OpenCV: 4.13.0


In [3]:
# Configuration
DATA_ROOT = Path(
    "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/"
    "CelebA_Spoof_/CelebA_Spoof"
)

ANALYSIS_SPLIT = "train"
SAMPLE_SIZE = 5000

SCRFD_DET_SIZE = (640, 640)
SCRFD_CONF_THRESH = 0.5

CELEBA_REFERENCE_FACTOR = 1.5
SCRFD_FACTOR_MIN = 1.00
SCRFD_FACTOR_MAX = 2.50
SCRFD_FACTOR_STEP = 0.05
SCRFD_FACTORS = np.round(
    np.arange(SCRFD_FACTOR_MIN, SCRFD_FACTOR_MAX + 1e-9, SCRFD_FACTOR_STEP),
    2,
).tolist()

# Reliable-overlap subset used ONLY to estimate equivalent crop factor.
RELIABLE_RAW_IOU_MIN = 0.50
RELIABLE_CENTER_SHIFT_MAX = 0.20

# TRAIN-only min-face candidates.
MIN_FACE_CANDIDATES = [32, 40, 48, 56, 60, 64, 72, 80, 96]

# Debug/project constraints, not benchmark standards.
MAX_TOTAL_TRAIN_DROP = 0.05
MAX_CLASS_TRAIN_DROP = 0.08
MAX_CLASS_DROP_GAP = 0.03

INPUT_SIZE_PAD = 224
SHOW_WORST_N = 12
SHOW_SMALL_FACE_N = 12
SAVE_RESULTS = True

# ============================================================
# Kaggle output controls
# ============================================================
COMPACT_OUTPUT = True

# Keep these False during normal runs to prevent notebook lag.
RENDER_INLINE_PLOTS = False
RENDER_GALLERIES = False
SHOW_VERBOSE_TABLES = False

# Plots are saved to disk instead of rendered inline.
SAVE_PLOTS = True
PLOT_DIR = Path("/kaggle/working/scrfd_analysis_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# tqdm output is useful but can also be disabled.
DISABLE_PROGRESS_BARS = False

JSON_PATH = DATA_ROOT / f"metas/intra_test/{ANALYSIS_SPLIT}_label.json"
assert DATA_ROOT.exists(), DATA_ROOT
assert JSON_PATH.exists(), JSON_PATH

with open(JSON_PATH, "r") as f:
    meta = json.load(f)

all_keys = list(meta.keys())
rng = np.random.default_rng(SEED)

if SAMPLE_SIZE is None or SAMPLE_SIZE >= len(all_keys):
    keys = all_keys
else:
    keys = rng.choice(all_keys, size=SAMPLE_SIZE, replace=False).tolist()

print("Analysis split            :", ANALYSIS_SPLIT)
print("Selected                  :", len(keys))
print("CelebA reference factor   :", CELEBA_REFERENCE_FACTOR)
print("Reliable raw IoU min      :", RELIABLE_RAW_IOU_MIN)
print("Reliable center shift max :", RELIABLE_CENTER_SHIFT_MAX)
print("Factor sweep              :", SCRFD_FACTORS[0], "→", SCRFD_FACTORS[-1])

Analysis split            : train
Selected                  : 5000
CelebA reference factor   : 1.5
Reliable raw IoU min      : 0.5
Reliable center shift max : 0.2
Factor sweep              : 1.0 → 2.5


In [4]:
# Initialize SCRFD-500M from buffalo_s
app = FaceAnalysis(
    name="buffalo_s",
    allowed_modules=["detection"],
    providers=["CPUExecutionProvider"],
)
app.prepare(ctx_id=-1, det_size=SCRFD_DET_SIZE, det_thresh=SCRFD_CONF_THRESH)

scrfd = app.models.get("detection")
if scrfd is None:
    raise RuntimeError("SCRFD detection model not found in buffalo_s")

print("SCRFD ready")

download_path: /root/.insightface/models/buffalo_s


100%|██████████| 124617/124617 [00:01<00:00, 99656.82KB/s] 


SCRFD ready


In [5]:
# Label / geometry helpers
CLASS_NAMES = {0: "Real", 1: "Physical Spoof", 2: "Digital Spoof"}

def label3_of(meta_value):
    raw = int(meta_value[40])
    if raw == 0:
        return 0
    if raw in [1,2,3,4,5,6,7]:
        return 1
    return 2

def read_celeba_bbox(img_path):
    bb_path = Path(str(img_path.with_suffix("")) + "_BB.txt")
    if not bb_path.exists():
        raise FileNotFoundError(bb_path)
    with open(bb_path, "r") as f:
        parts = f.readline().strip().split()
    if len(parts) < 4:
        raise ValueError(f"Invalid bbox file: {bb_path}")
    x, y, w, h = map(float, parts[:4])
    image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(img_path)
    H, W = image.shape[:2]
    x = x / 224.0 * W
    y = y / 224.0 * H
    w = w / 224.0 * W
    h = h / 224.0 * H
    if w <= 0 or h <= 0:
        raise ValueError(f"Non-positive CelebA bbox: {(x,y,w,h)}")
    return np.array([x,y,x+w,y+h], dtype=np.float64), image

def bbox_area(b):
    x1,y1,x2,y2 = map(float,b)
    return max(0.0,x2-x1)*max(0.0,y2-y1)

def bbox_wh(b):
    x1,y1,x2,y2 = map(float,b)
    return np.array([x2-x1,y2-y1], dtype=np.float64)

def bbox_center(b):
    x1,y1,x2,y2 = map(float,b)
    return np.array([(x1+x2)/2.0,(y1+y2)/2.0], dtype=np.float64)

def bbox_intersection_area(a,b):
    ax1,ay1,ax2,ay2 = map(float,a)
    bx1,by1,bx2,by2 = map(float,b)
    ix1=max(ax1,bx1); iy1=max(ay1,by1)
    ix2=min(ax2,bx2); iy2=min(ay2,by2)
    return max(0.0,ix2-ix1)*max(0.0,iy2-iy1)

def bbox_iou(a,b):
    inter=bbox_intersection_area(a,b)
    union=bbox_area(a)+bbox_area(b)-inter
    return 0.0 if union<=0 else inter/union

def reference_coverage(reference_box,candidate_box):
    inter=bbox_intersection_area(reference_box,candidate_box)
    return inter/max(bbox_area(reference_box),1e-8)

def candidate_precision(reference_box,candidate_box):
    inter=bbox_intersection_area(reference_box,candidate_box)
    return inter/max(bbox_area(candidate_box),1e-8)

def expand_square_bbox(b,factor):
    x1,y1,x2,y2 = map(float,b)
    w=x2-x1; h=y2-y1
    side=max(w,h)*float(factor)
    cx=(x1+x2)/2.0; cy=(y1+y2)/2.0
    return np.array([cx-side/2,cy-side/2,cx+side/2,cy+side/2], dtype=np.float64)

def normalized_center_shift(gt,det):
    gt_wh=bbox_wh(gt)
    denom=max(gt_wh[0],gt_wh[1],1e-8)
    return float(np.linalg.norm(bbox_center(gt)-bbox_center(det))/denom)

def select_scrfd_match(bboxes,gt_bbox):
    if bboxes is None or len(bboxes)==0:
        return None,None,0
    ious=[bbox_iou(row[:4],gt_bbox) for row in bboxes]
    idx=int(np.argmax(ious))
    return np.asarray(bboxes[idx][:4],dtype=np.float64), float(bboxes[idx][4]), len(bboxes)

In [6]:
# Run SCRFD once and cache geometry
CACHE_PATH = Path(
    f"/kaggle/working/scrfd_geometry_{ANALYSIS_SPLIT}_{len(keys)}_seed{SEED}.csv"
)
USE_CACHE = True

if USE_CACHE and CACHE_PATH.exists():
    print("Loading cache:", CACHE_PATH)
    df = pd.read_csv(CACHE_PATH)
else:
    rows=[]; errors=[]
    for key in tqdm(keys, desc="SCRFD geometry", disable=DISABLE_PROGRESS_BARS):
        img_path=DATA_ROOT/key
        try:
            gt,image=read_celeba_bbox(img_path)
            H,W=image.shape[:2]
            gt_w,gt_h=bbox_wh(gt)
            label3=label3_of(meta[key])
            base={
                "key":key,
                "label3":label3,
                "class_name":CLASS_NAMES[label3],
                "image_w":W,"image_h":H,
                "gt_x1":gt[0],"gt_y1":gt[1],"gt_x2":gt[2],"gt_y2":gt[3],
                "gt_min_side":min(gt_w,gt_h),
                "gt_max_side":max(gt_w,gt_h),
            }
            bboxes,_=scrfd.detect(image,max_num=0,metric="default")
            det,conf,n_faces=select_scrfd_match(bboxes,gt)
            base["n_scrfd_faces"]=int(n_faces)
            base["detected"]=det is not None
            if det is None:
                rows.append(base); continue
            dt_w,dt_h=bbox_wh(det)
            base.update({
                "scrfd_conf":conf,
                "det_x1":det[0],"det_y1":det[1],"det_x2":det[2],"det_y2":det[3],
                "det_width":dt_w,"det_height":dt_h,
                "det_min_side":min(dt_w,dt_h),
                "det_max_side":max(dt_w,dt_h),
                "raw_iou":bbox_iou(gt,det),
                "center_shift_norm":normalized_center_shift(gt,det),
                "scale_ratio_det_over_gt":max(dt_w,dt_h)/max(max(gt_w,gt_h),1e-8),
            })
            rows.append(base)
        except Exception as exc:
            errors.append((key,repr(exc)))
    df=pd.DataFrame(rows)
    df.to_csv(CACHE_PATH,index=False)
    print("Saved cache:",CACHE_PATH)
    print("Errors:",len(errors))

print("Rows:", len(df))
if not COMPACT_OUTPUT:
    display(df.head())

SCRFD geometry:   0%|          | 0/5000 [00:00<?, ?it/s]

Saved cache: /kaggle/working/scrfd_geometry_train_5000_seed42.csv
Errors: 0
Rows: 5000


In [7]:
# Detection coverage — keep ALL samples for analysis
n_total=len(df)
n_detected=int(df["detected"].sum())
n_failed=n_total-n_detected
failure_rate=n_failed/max(n_total,1)

print(f"Total samples      : {n_total}")
print(f"SCRFD detected     : {n_detected}")
print(f"SCRFD no detection : {n_failed}")
print(f"Failure rate       : {100*failure_rate:.3f}%")

coverage_by_class=df.groupby("class_name").agg(N=("key","size"),detected=("detected","sum"))
coverage_by_class["failure_%"]=100*(1-coverage_by_class["detected"]/coverage_by_class["N"])
if SHOW_VERBOSE_TABLES:
    display(coverage_by_class)
else:
    print("\nFailure by class:")
    for cls, row in coverage_by_class.iterrows():
        print(
            f"  {cls:15s}  N={int(row['N']):5d}  "
            f"failure={row['failure_%']:.2f}%"
        )

d=df[df["detected"]].copy()

Total samples      : 5000
SCRFD detected     : 4745
SCRFD no detection : 255
Failure rate       : 5.100%

Failure by class:
  Digital Spoof    N= 1031  failure=4.36%
  Physical Spoof   N= 2301  failure=8.47%
  Real             N= 1668  failure=0.90%


In [8]:
# Reliable overlap subset for factor estimation
reliable=d[
    (d["raw_iou"]>=RELIABLE_RAW_IOU_MIN)
    &
    (d["center_shift_norm"]<=RELIABLE_CENTER_SHIFT_MAX)
].copy()
unreliable=d.drop(reliable.index).copy()

print("Detected samples:",len(d))
print("Reliable pairs  :",len(reliable),f"({100*len(reliable)/max(len(d),1):.2f}%)")
print("Excluded pairs  :",len(unreliable),f"({100*len(unreliable)/max(len(d),1):.2f}%)")

if SHOW_VERBOSE_TABLES:
    display(
        reliable[
            ["raw_iou", "center_shift_norm", "scale_ratio_det_over_gt"]
        ].describe(
            percentiles=[.01,.05,.10,.25,.5,.75,.90,.95,.99]
        )
    )
else:
    print(
        "Reliable medians | "
        f"raw IoU={reliable.raw_iou.median():.3f} | "
        f"center shift={reliable.center_shift_norm.median():.3f} | "
        f"scale ratio={reliable.scale_ratio_det_over_gt.median():.3f}"
    )

Detected samples: 4745
Reliable pairs  : 4738 (99.85%)
Excluded pairs  : 7 (0.15%)
Reliable medians | raw IoU=0.875 | center shift=0.026 | scale ratio=0.963


In [9]:
# Fine factor sweep on RELIABLE pairs only
factor_rows=[]
best_factors_per_sample=[]

for idx,row in tqdm(reliable.iterrows(), total=len(reliable), desc="Per-sample best factor", disable=DISABLE_PROGRESS_BARS):
    gt=np.array([row.gt_x1,row.gt_y1,row.gt_x2,row.gt_y2],dtype=np.float64)
    det=np.array([row.det_x1,row.det_y1,row.det_x2,row.det_y2],dtype=np.float64)
    reference=expand_square_bbox(gt,CELEBA_REFERENCE_FACTOR)
    scores=[]
    for factor in SCRFD_FACTORS:
        candidate=expand_square_bbox(det,factor)
        scores.append((factor,bbox_iou(reference,candidate)))
    best_factor,best_iou=max(scores,key=lambda x:x[1])
    best_factors_per_sample.append({"index":idx,"best_factor":best_factor,"best_iou":best_iou})

for factor in SCRFD_FACTORS:
    ious=[]; coverages=[]; precisions=[]; area_ratios=[]
    for _,row in reliable.iterrows():
        gt=np.array([row.gt_x1,row.gt_y1,row.gt_x2,row.gt_y2],dtype=np.float64)
        det=np.array([row.det_x1,row.det_y1,row.det_x2,row.det_y2],dtype=np.float64)
        reference=expand_square_bbox(gt,CELEBA_REFERENCE_FACTOR)
        candidate=expand_square_bbox(det,factor)
        ious.append(bbox_iou(reference,candidate))
        coverages.append(reference_coverage(reference,candidate))
        precisions.append(candidate_precision(reference,candidate))
        area_ratios.append(bbox_area(candidate)/max(bbox_area(reference),1e-8))
    ious=np.asarray(ious); coverages=np.asarray(coverages)
    precisions=np.asarray(precisions); area_ratios=np.asarray(area_ratios)
    factor_rows.append({
        "scrfd_factor":factor,
        "mean_iou":np.mean(ious),
        "median_iou":np.median(ious),
        "p10_iou":np.quantile(ious,.10),
        "p05_iou":np.quantile(ious,.05),
        "median_ref_coverage":np.median(coverages),
        "p10_ref_coverage":np.quantile(coverages,.10),
        "median_candidate_precision":np.median(precisions),
        "median_area_ratio":np.median(area_ratios),
        "fraction_iou_ge_090":np.mean(ious>=.90),
        "fraction_iou_lt_070":np.mean(ious<.70),
    })

factor_table = pd.DataFrame(factor_rows)

if SHOW_VERBOSE_TABLES:
    display(factor_table.round(4))
else:
    print(f"Factor sweep complete: {len(factor_table)} candidates")

Per-sample best factor:   0%|          | 0/4738 [00:00<?, ?it/s]

Factor sweep complete: 31 candidates


In [10]:
# Geometric factor candidates
best_iou_row = factor_table.sort_values(
    ["median_iou", "p10_iou", "mean_iou"],
    ascending=False,
).iloc[0]

BEST_IOU_FACTOR = float(best_iou_row["scrfd_factor"])

# Smallest factor that covers almost all reference context for typical/reasonably hard samples.
# Heuristic only; PAD inference must validate the final choice.
safe = factor_table[
    (factor_table["median_ref_coverage"] >= .97)
    &
    (factor_table["p10_ref_coverage"] >= .90)
].copy()

CONTEXT_SAFE_FACTOR = (
    float(safe.sort_values("scrfd_factor").iloc[0]["scrfd_factor"])
    if len(safe)
    else None
)

print("Best median-IoU factor:", BEST_IOU_FACTOR)
print("Context-safe factor   :", CONTEXT_SAFE_FACTOR)

# Print only a small neighborhood around the optimum.
ranked = factor_table.copy()
ranked["distance_to_best"] = (
    ranked["scrfd_factor"] - BEST_IOU_FACTOR
).abs()

preview = (
    ranked
    .sort_values(["distance_to_best", "scrfd_factor"])
    .head(7)
    .sort_values("scrfd_factor")
    [[
        "scrfd_factor",
        "median_iou",
        "p10_iou",
        "median_ref_coverage",
        "p10_ref_coverage",
        "median_area_ratio",
    ]]
)

print("\nFactor neighborhood around optimum:")
print(preview.round(4).to_string(index=False))

plt.figure(figsize=(9,5))
plt.plot(
    factor_table.scrfd_factor,
    factor_table.median_iou,
    marker="o",
    label="Median IoU",
)
plt.plot(
    factor_table.scrfd_factor,
    factor_table.p10_iou,
    marker="o",
    label="P10 IoU",
)
plt.plot(
    factor_table.scrfd_factor,
    factor_table.median_ref_coverage,
    marker="o",
    label="Median ref coverage",
)
plt.axvline(1.5, linestyle="--", label="Old factor 1.5×")
plt.axvline(
    BEST_IOU_FACTOR,
    linestyle=":",
    label=f"Best-IoU {BEST_IOU_FACTOR:.2f}×",
)
plt.xlabel("SCRFD expansion factor")
plt.ylabel("Geometry score")
plt.title("SCRFD crop vs CelebA 1.5× reference context")
plt.grid(alpha=.25)
plt.legend()
plt.tight_layout()

factor_plot_path = PLOT_DIR / "crop_factor_sweep.png"

if SAVE_PLOTS:
    plt.savefig(factor_plot_path, dpi=160, bbox_inches="tight")

if RENDER_INLINE_PLOTS:
    plt.show()
else:
    plt.close()

if SAVE_PLOTS:
    print("Saved plot:", factor_plot_path)


Best median-IoU factor: 1.55
Context-safe factor   : 1.65

Factor neighborhood around optimum:
 scrfd_factor  median_iou  p10_iou  median_ref_coverage  p10_ref_coverage  median_area_ratio
         1.40      0.8075   0.6843               0.8082            0.6845             0.8084
         1.45      0.8617   0.7328               0.8667            0.7342             0.8672
         1.50      0.9018   0.7788               0.9239            0.7857             0.9280
         1.55      0.9155   0.8021               0.9705            0.8388             0.9909
         1.60      0.9072   0.7945               0.9971            0.8913             1.0559
         1.65      0.8778   0.7580               1.0000            0.9388             1.1229
         1.70      0.8351   0.7181               1.0000            0.9750             1.1920
Saved plot: /kaggle/working/scrfd_analysis_plots/crop_factor_sweep.png


In [11]:
# Per-sample optimal-factor distribution
best_factor_df = pd.DataFrame(best_factors_per_sample)

factor_dist = (
    best_factor_df.best_factor
    .value_counts()
    .sort_index()
)

factor_dist_df = pd.DataFrame(
    {
        "factor": factor_dist.index,
        "N": factor_dist.values,
    }
)

factor_dist_df["percent"] = (
    100 * factor_dist_df.N / len(best_factor_df)
)

median_per_sample_factor = float(
    best_factor_df.best_factor.median()
)

print(
    "Median per-sample best factor:",
    median_per_sample_factor,
)

# Only print the most common factors.
top_factor_dist = (
    factor_dist_df
    .sort_values("percent", ascending=False)
    .head(8)
)

print("\nMost common per-sample best factors:")
print(
    top_factor_dist
    .round({"factor": 2, "percent": 2})
    .to_string(index=False)
)

plt.figure(figsize=(12,4))
plt.bar(
    factor_dist_df.factor.astype(str),
    factor_dist_df.percent,
)
plt.xticks(rotation=70)
plt.xlabel("Per-sample best SCRFD factor")
plt.ylabel("Reliable samples (%)")
plt.title("Distribution of per-sample optimal crop expansion")
plt.grid(axis="y", alpha=.25)
plt.tight_layout()

dist_plot_path = PLOT_DIR / "per_sample_best_factor_distribution.png"

if SAVE_PLOTS:
    plt.savefig(dist_plot_path, dpi=160, bbox_inches="tight")

if RENDER_INLINE_PLOTS:
    plt.show()
else:
    plt.close()

if SAVE_PLOTS:
    print("Saved plot:", dist_plot_path)


Median per-sample best factor: 1.55

Most common per-sample best factors:
 factor    N  percent
   1.55 1042    21.99
   1.60  884    18.66
   1.50  823    17.37
   1.65  569    12.01
   1.45  458     9.67
   1.70  309     6.52
   1.40  179     3.78
   1.75  162     3.42
Saved plot: /kaggle/working/scrfd_analysis_plots/per_sample_best_factor_distribution.png


In [12]:
# SCRFD face-size distribution — NO filtering yet
face_sizes = d[
    ["det_min_side", "det_max_side", "class_name"]
].copy()

q = face_sizes.det_min_side.quantile(
    [0.01, 0.02, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99]
)

print("Detected min-face-side quantiles [px]:")
for quantile, value in q.items():
    print(f"  q{int(quantile*100):02d}: {value:.1f}")

plt.figure(figsize=(10,5))

for cls in CLASS_NAMES.values():
    vals = face_sizes[
        face_sizes.class_name == cls
    ].det_min_side

    plt.hist(
        vals,
        bins=70,
        alpha=.40,
        density=True,
        label=cls,
    )

plt.xlabel("SCRFD min(face width, face height) [px]")
plt.ylabel("Density")
plt.title("Native detected-face size distribution")
plt.xlim(left=0)
plt.grid(alpha=.2)
plt.legend()
plt.tight_layout()

face_size_plot_path = PLOT_DIR / "face_size_distribution.png"

if SAVE_PLOTS:
    plt.savefig(
        face_size_plot_path,
        dpi=160,
        bbox_inches="tight",
    )

if RENDER_INLINE_PLOTS:
    plt.show()
else:
    plt.close()

if SAVE_PLOTS:
    print("Saved plot:", face_size_plot_path)


Detected min-face-side quantiles [px]:
  q01: 49.6
  q02: 58.2
  q05: 74.4
  q10: 91.2
  q50: 168.7
  q90: 266.3
  q95: 298.4
  q99: 515.9
Saved plot: /kaggle/working/scrfd_analysis_plots/face_size_distribution.png


In [13]:
# TRAIN-only minimum-face threshold analysis
threshold_rows=[]
for threshold in MIN_FACE_CANDIDATES:
    temp=d.copy()
    temp["drop_small"]=temp.det_min_side < threshold
    total_drop=float(temp.drop_small.mean())
    class_drop=temp.groupby("class_name").drop_small.mean().to_dict()
    drop_values=[float(class_drop.get(cls,0.0)) for cls in CLASS_NAMES.values()]
    max_class_drop=max(drop_values)
    class_drop_gap=max(drop_values)-min(drop_values)

    approx_crop_side=threshold*BEST_IOU_FACTOR
    approx_resize_upscale=INPUT_SIZE_PAD/max(approx_crop_side,1e-8)

    threshold_rows.append({
        "min_face_px":threshold,
        "total_drop_rate":total_drop,
        "real_drop_rate":float(class_drop.get("Real",0.0)),
        "physical_drop_rate":float(class_drop.get("Physical Spoof",0.0)),
        "digital_drop_rate":float(class_drop.get("Digital Spoof",0.0)),
        "max_class_drop_rate":max_class_drop,
        "class_drop_gap":class_drop_gap,
        "approx_crop_side_px":approx_crop_side,
        "approx_resize_upscale_to_224":approx_resize_upscale,
    })

minface_table=pd.DataFrame(threshold_rows)
view=minface_table.copy()
for c in ["total_drop_rate","real_drop_rate","physical_drop_rate","digital_drop_rate","max_class_drop_rate","class_drop_gap"]:
    view[c]=100*view[c]
view=view.rename(columns={
    "total_drop_rate":"total_drop_%",
    "real_drop_rate":"real_drop_%",
    "physical_drop_rate":"physical_drop_%",
    "digital_drop_rate":"digital_drop_%",
    "max_class_drop_rate":"max_class_drop_%",
    "class_drop_gap":"class_drop_gap_pp",
})
compact_cols = [
    "min_face_px",
    "total_drop_%",
    "real_drop_%",
    "physical_drop_%",
    "digital_drop_%",
    "class_drop_gap_pp",
    "approx_resize_upscale_to_224",
]

print("\nMin-face candidate summary:")
print(
    view[compact_cols]
    .round(3)
    .to_string(index=False)
)


Min-face candidate summary:
 min_face_px  total_drop_%  real_drop_%  physical_drop_%  digital_drop_%  class_drop_gap_pp  approx_resize_upscale_to_224
          32         0.190        0.060            0.000           0.811              0.811                         4.516
          40         0.506        0.726            0.000           1.217              1.217                         3.613
          48         0.906        1.331            0.000           2.130              2.130                         3.011
          56         1.665        3.267            0.000           2.535              3.267                         2.581
          60         2.171        4.356            0.142           2.840              4.213                         2.409
          64         2.677        5.384            0.285           3.245              5.099                         2.258
          72         4.320        9.195            0.380           4.564              8.816                         2

In [14]:
# Suggest TRAIN min-face threshold
eligible=minface_table[
    (minface_table.total_drop_rate<=MAX_TOTAL_TRAIN_DROP)
    &
    (minface_table.max_class_drop_rate<=MAX_CLASS_TRAIN_DROP)
    &
    (minface_table.class_drop_gap<=MAX_CLASS_DROP_GAP)
].copy()

SUGGESTED_MIN_FACE=int(eligible.sort_values("min_face_px").iloc[-1].min_face_px) if len(eligible) else None

print("Suggested TRAIN-only min face:",SUGGESTED_MIN_FACE)
print(f"Constraints: overall drop <= {100*MAX_TOTAL_TRAIN_DROP:.1f}% | any class <= {100*MAX_CLASS_TRAIN_DROP:.1f}% | class gap <= {100*MAX_CLASS_DROP_GAP:.1f} pp")
if SUGGESTED_MIN_FACE is not None:
    row = minface_table[
        minface_table.min_face_px == SUGGESTED_MIN_FACE
    ].iloc[0]

    print(
        "Selected threshold stats | "
        f"overall drop={100*row.total_drop_rate:.2f}% | "
        f"real={100*row.real_drop_rate:.2f}% | "
        f"physical={100*row.physical_drop_rate:.2f}% | "
        f"digital={100*row.digital_drop_rate:.2f}% | "
        f"upscale≈{row.approx_resize_upscale_to_224:.2f}×"
    )
else:
    print("No candidate satisfies all heuristic constraints; inspect table manually.")

Suggested TRAIN-only min face: 48
Constraints: overall drop <= 5.0% | any class <= 8.0% | class gap <= 3.0 pp
Selected threshold stats | overall drop=0.91% | real=1.33% | physical=0.00% | digital=2.13% | upscale≈3.01×


In [15]:
# Optional gallery — smallest detected faces
if not RENDER_GALLERIES:
    print(
        "Small-face gallery skipped "
        "(RENDER_GALLERIES=False)."
    )
else:
    smallest = (
        d
        .sort_values("det_min_side")
        .head(SHOW_SMALL_FACE_N)
    )

    cols = 4
    rows_n = math.ceil(len(smallest) / cols)

    fig, axes = plt.subplots(
        rows_n,
        cols,
        figsize=(14, 3.3 * rows_n),
    )

    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes,
        smallest.iterrows(),
    ):
        img = cv2.imread(
            str(DATA_ROOT / row["key"])
        )
        img = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB,
        )

        x1, y1, x2, y2 = [
            int(row[k])
            for k in [
                "det_x1",
                "det_y1",
                "det_x2",
                "det_y2",
            ]
        ]

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2,
        )

        ax.imshow(img)
        ax.set_title(
            f"{row.class_name}\\n"
            f"min side={row.det_min_side:.1f}px"
        )
        ax.axis("off")

    plt.tight_layout()

    gallery_path = PLOT_DIR / "smallest_faces_gallery.png"

    if SAVE_PLOTS:
        plt.savefig(
            gallery_path,
            dpi=130,
            bbox_inches="tight",
        )

    if RENDER_INLINE_PLOTS:
        plt.show()
    else:
        plt.close()

    print("Saved gallery:", gallery_path)


Small-face gallery skipped (RENDER_GALLERIES=False).


In [16]:
# Optional gallery — unreliable CelebA/SCRFD pairs
if not RENDER_GALLERIES:
    print(
        "Unreliable-pair gallery skipped "
        "(RENDER_GALLERIES=False)."
    )
else:
    worst = (
        unreliable
        .sort_values(
            ["raw_iou", "center_shift_norm"],
            ascending=[True, False],
        )
        .head(SHOW_WORST_N)
    )

    if len(worst) == 0:
        print(
            "No unreliable matches "
            "under current criteria."
        )
    else:
        cols = 4
        rows_n = math.ceil(len(worst) / cols)

        fig, axes = plt.subplots(
            rows_n,
            cols,
            figsize=(14, 3.3 * rows_n),
        )

        axes = np.array(axes).reshape(-1)

        for ax in axes:
            ax.axis("off")

        for ax, (_, row) in zip(
            axes,
            worst.iterrows(),
        ):
            img = cv2.imread(
                str(DATA_ROOT / row["key"])
            )
            img = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2RGB,
            )

            gt = np.array([
                row.gt_x1,
                row.gt_y1,
                row.gt_x2,
                row.gt_y2,
            ])

            det = np.array([
                row.det_x1,
                row.det_y1,
                row.det_x2,
                row.det_y2,
            ])

            for box, color, label in [
                (gt, (0,255,0), "CelebA"),
                (det, (255,0,0), "SCRFD"),
            ]:
                x1, y1, x2, y2 = [
                    int(round(v))
                    for v in box
                ]

                cv2.rectangle(
                    img,
                    (x1, y1),
                    (x2, y2),
                    color,
                    2,
                )

                cv2.putText(
                    img,
                    label,
                    (
                        max(0, x1),
                        max(20, y1 - 5),
                    ),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    .5,
                    color,
                    2,
                )

            ax.imshow(img)
            ax.set_title(
                f"IoU={row.raw_iou:.2f}\\n"
                f"center={row.center_shift_norm:.2f}"
            )
            ax.axis("off")

        plt.tight_layout()

        gallery_path = (
            PLOT_DIR
            /
            "unreliable_bbox_pairs_gallery.png"
        )

        if SAVE_PLOTS:
            plt.savefig(
                gallery_path,
                dpi=130,
                bbox_inches="tight",
            )

        if RENDER_INLINE_PLOTS:
            plt.show()
        else:
            plt.close()

        print("Saved gallery:", gallery_path)


Unreliable-pair gallery skipped (RENDER_GALLERIES=False).


In [17]:
# Detection failure rate by GT face size — no samples removed
size_bins = [
    0, 32, 48, 60, 80, 120, 180, 240, 320, np.inf
]

size_labels = [
    "<32",
    "32–47",
    "48–59",
    "60–79",
    "80–119",
    "120–179",
    "180–239",
    "240–319",
    ">=320",
]

df_size = df.copy()

df_size["gt_size_bin"] = pd.cut(
    df_size.gt_min_side,
    bins=size_bins,
    labels=size_labels,
    right=False,
)

failure_by_size = (
    df_size
    .groupby(
        "gt_size_bin",
        observed=False,
    )
    .agg(
        N=("key", "size"),
        detected=("detected", "sum"),
    )
)

failure_by_size["failure_%"] = (
    100
    *
    (
        1
        -
        failure_by_size.detected
        /
        failure_by_size.N.clip(lower=1)
    )
)

print("\nDetection failure by GT face-size bin:")
print(
    failure_by_size
    .round(2)
    .to_string()
)

plt.figure(figsize=(10,4))
plt.bar(
    failure_by_size.index.astype(str),
    failure_by_size["failure_%"],
)
plt.xticks(rotation=45)
plt.ylabel("SCRFD detection failure (%)")
plt.xlabel("CelebA GT min face side [px]")
plt.title(
    "Detector failure vs face size — no samples removed"
)
plt.grid(axis="y", alpha=.25)
plt.tight_layout()

failure_plot_path = (
    PLOT_DIR
    /
    "detection_failure_by_face_size.png"
)

if SAVE_PLOTS:
    plt.savefig(
        failure_plot_path,
        dpi=160,
        bbox_inches="tight",
    )

if RENDER_INLINE_PLOTS:
    plt.show()
else:
    plt.close()

if SAVE_PLOTS:
    print("Saved plot:", failure_plot_path)



Detection failure by GT face-size bin:
                N  detected  failure_%
gt_size_bin                           
<32            13        13       0.00
32–47          45        44       2.22
48–59          66        66       0.00
60–79         239       230       3.77
80–119        801       772       3.62
120–179      1803      1785       1.00
180–239      1235      1216       1.54
240–319       575       499      13.22
>=320         223       120      46.19
Saved plot: /kaggle/working/scrfd_analysis_plots/detection_failure_by_face_size.png


In [18]:
# Final recommendation summary
print("=== PREPROCESS POLICY ANALYSIS ===")
print(f"Reliable geometry pairs: {len(reliable)} / {len(d)} detected ({100*len(reliable)/max(len(d),1):.2f}%)")
print(f"Best median-IoU SCRFD factor : {BEST_IOU_FACTOR:.2f}×")
print(f"Context-safe SCRFD factor    : {CONTEXT_SAFE_FACTOR}")
print(f"Suggested TRAIN-only min face: {SUGGESTED_MIN_FACE} px")

print("\nRecommended workflow:")
print("1. Freeze crop factor using TRAIN/VAL geometry only.")
print("2. Build full SCRFD cache.")
print("3. TRAIN: keep valid detections with min(width,height) >= chosen threshold.")
print("4. TEST: do NOT drop small faces; report detection coverage/failures separately.")
print("5. Before full retraining, run causal PAD check: same image/model, CelebA-1.5 crop vs SCRFD-old-factor vs SCRFD-new-factor.")
print("6. Freeze one identical crop policy for E1/E2/E3.")

=== PREPROCESS POLICY ANALYSIS ===
Reliable geometry pairs: 4738 / 4745 detected (99.85%)
Best median-IoU SCRFD factor : 1.55×
Context-safe SCRFD factor    : 1.65
Suggested TRAIN-only min face: 48 px

Recommended workflow:
1. Freeze crop factor using TRAIN/VAL geometry only.
2. Build full SCRFD cache.
3. TRAIN: keep valid detections with min(width,height) >= chosen threshold.
4. TEST: do NOT drop small faces; report detection coverage/failures separately.
5. Before full retraining, run causal PAD check: same image/model, CelebA-1.5 crop vs SCRFD-old-factor vs SCRFD-new-factor.
6. Freeze one identical crop policy for E1/E2/E3.


In [19]:
# Save outputs / candidate policy
if SAVE_RESULTS:
    out_geometry=Path("/kaggle/working/scrfd_crop_factor_analysis.csv")
    out_minface=Path("/kaggle/working/scrfd_min_face_threshold_analysis.csv")
    out_policy=Path("/kaggle/working/scrfd_preprocess_policy_candidate.json")

    factor_table.to_csv(out_geometry,index=False)
    minface_table.to_csv(out_minface,index=False)

    policy={
        "analysis_split":ANALYSIS_SPLIT,
        "sample_size":len(df),
        "detector":"SCRFD buffalo_s detection",
        "det_size":list(SCRFD_DET_SIZE),
        "det_threshold":SCRFD_CONF_THRESH,
        "reference_crop":{"source":"CelebA-Spoof _BB.txt","factor":CELEBA_REFERENCE_FACTOR},
        "reliable_match_rule":{"raw_iou_min":RELIABLE_RAW_IOU_MIN,"center_shift_max":RELIABLE_CENTER_SHIFT_MAX},
        "crop_factor_candidates":{"best_median_iou":BEST_IOU_FACTOR,"context_safe":CONTEXT_SAFE_FACTOR},
        "train_min_face_candidate_px":SUGGESTED_MIN_FACE,
        "test_policy":"Do not remove small faces; report detector coverage/failures separately.",
    }
    with open(out_policy,"w") as f:
        json.dump(policy,f,indent=2)

    print("\nSaved analysis artifacts:")
    for p in [
        out_geometry,
        out_minface,
        out_policy,
        CACHE_PATH,
    ]:
        print(" ", p)

    if SAVE_PLOTS:
        print(" Plot directory:", PLOT_DIR)


Saved analysis artifacts:
  /kaggle/working/scrfd_crop_factor_analysis.csv
  /kaggle/working/scrfd_min_face_threshold_analysis.csv
  /kaggle/working/scrfd_preprocess_policy_candidate.json
  /kaggle/working/scrfd_geometry_train_5000_seed42.csv
 Plot directory: /kaggle/working/scrfd_analysis_plots
